In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("./papers/202_Data_Leakage_in_Federated_.pdf") 
#loader= PyMuPDFDirectoryLoader("./papers/")  # or DirectoryLoader for folder
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
#from langchain_community.graph_transformers import LLMGraphTransformer
from langchain_ollama import ChatOllama
from langchain_neo4j import Neo4jGraph

llm = ChatOllama(model="llama3.1:8b", temperature=0)
graph_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Concept", "Method", "Finding", "Paper"],
    allowed_relationships=["RELATES_TO", "USES", "CONTRADICTS", "BUILDS_ON"]
)

graph_docs = graph_transformer.convert_to_graph_documents(chunks)

graph = Neo4jGraph(url="bolt://localhost:7687", username="neo4j", password="pleaseletmein")
graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=True)

# Add vector index for hybrid RAG
from langchain_neo4j import Neo4jVector
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    url="bolt://localhost:7687",
    username="neo4j",
    password="pleaseletmein",
    index_name="paper_vector",
    node_label="Chunk",  # or your entity labels
    text_node_properties=["text"]
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Summarize this research section as an Obsidian note.
Include: title, key concepts (as [[wikilink]]), findings, relations to other papers.
Output pure Markdown with frontmatter.
---
title: ...
tags: [paper, research]
---

# Executive Summary
...

**Connected Concepts:** [[ConceptA]] [[MethodB]]
""")

# Run on each chunk or aggregated paper
# Save to vault:
with open("~/research-vault/paper-title.md", "w") as f:
    f.write(note_md)